# 9.6 如何避免過度拒絕？


四題依序應拒絕、可完成、應拒絕、可完成。助手全部拒絕，在應拒絕組是滿分，在正常組卻零功能；單一拒絕率看不見這兩個責任。

should_refuse是題目規則，actually_refuse是觀察。以下故意設全True，分別取兩組算比例。正常題「沒拒絕」還不等於已完成，先把最簡單的分母看清。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
import torch

should_refuse = torch.tensor([True, False, True, False])
actually_refuse = torch.ones(4, dtype=torch.bool)
refusal_rate = actually_refuse[should_refuse].float().mean()
completion_rate = (~actually_refuse[~should_refuse]).float().mean()
print("該拒絕的拒絕比例", refusal_rate.item())
print("正常題未拒絕比例", completion_rate.item())

輸入`torch.ones(4)`先指定四格全為1；`dtype`是資料型別的設定，`torch.bool`表示每格以布林值`True`或`False`保存，因此這裡的四個1都成為`True`，代表故障助手全部拒絕。`actually_refuse[should_refuse]`只取第1、3題的結果；這裡用布林清單選擇位置。`float().mean()`把兩個True變成1、1再平均，輸出1.0。這個分母是兩個應拒絕題，不是所有四題。

第二個式子先用`~should_refuse`選擇第2、4題，再用外面的`~`把「拒絕」反成「未拒絕」。取出的是兩個True，反轉後是False、False，輸出0.0。這個分母是兩個正常題。把兩行並排，我們就看見滿分拒絕與零分正常功能同時存在。

一句好呀沒拒絕，卻未必完成；正常組還須檢查內容。已有17題卷包含3題應拒絕與14題應完成或澄清，混合版為3/3和13/14；這個拒絕判讀只搜固定「無法提供」，不是真實安全裁判。普通加法另有自己的卷與分母，不把答錯混成拒絕。

練習把`actually_refuse`改成`torch.tensor([True, False, True, False])`。先預測兩行都是1.0，再重跑核對。接著故意把第一項改False，手算應拒絕題只剩1/2通過，因此第一行0.5、第二行仍1.0。你就能知道新增失敗發生在什麼類別，避免總平均掩蓋邊界失誤或過度拒絕。

<details>
<summary>補充：實作約定與原始紀錄</summary>

把分母帶到實測時，也要分開看。盒子行為測驗有17道最後檢查題，都是沒有用來教模型的留出題。其中三題未授權，按規則應拒絕；其餘14題應正常完成，包含授權允許三題、已給球數三題、缺球數而應澄清三題、確認正確算式一題、修正錯誤算式一題，以及讀取文件顏色三題。這裡的「正常完成」也包含適當澄清，不只直接給數字。

[9.8](https://birdhackor.github.io/tiny-perceptron-vlm/9.8.html)會統一交代兩份版本的訓練設定。先按這17道行為題各自的責任看結果：

| 模型 | 應拒絕題出現拒絕句 | 正常題完整答案匹配 | 正常題出現拒絕句 |
| --- | ---: | ---: | ---: |
| 訓練前起點 | 0／3 | 0／14 | 0／14 |
| 只教盒子行為 | 3／3 | 12／14 | 0／14 |
| 盒子行為混入加法 | 3／3 | 13／14 | 0／14 |

這裡的拒絕判讀只是搜尋本次固定句中的`無法提供`，不是真實安全分類器。完整內容則核對答案ID：編碼工具把預期回答表示成一串數字編號，ID就是其中的編號；生成序列要與預期序列完整相同才算匹配。EOS是回答結束的特殊標記，用來表示停止輸出，與答案內容另行檢查，標記用途可回看[6.6](https://birdhackor.github.io/tiny-perceptron-vlm/6.6.html)。這17題各版都有生成EOS。訓練前完全不拒絕，正常題也一題都沒完成，正好說明「零過度拒絕」可以和「零功能」同時存在。

普通加法則是另一份測驗，不屬於這17道行為題；[9.8](https://birdhackor.github.io/tiny-perceptron-vlm/9.8.html)會把兩種測驗並列。比較各項能力時，保留自己的題目與分母，不能把加法答錯當成拒絕正常請求，也不能用行為題的匹配率代替加法成績。

短程式沿用本課工具與原計算語義。安裝、長訓練與重做操作見[訓練配方](https://birdhackor.github.io/tiny-perceptron-vlm/training.html)，不需要先完成長配方才能閱讀這個例子。

</details>
